# RAG Document Assistant — Day 3: The Full RAG Chain
### Connecting retrieval to generation — your first real, grounded answers.

**Stack:** Google Gemini 2.5 Flash + Gemini Embeddings + LangChain + ChromaDB

**Recap of Day 2:** we embedded every chunk from `data/`, stored them permanently in a Chroma vector store at `chroma_db/`, and wrote a `retrieve()` function that returns the top-k most relevant chunks for a query — pure vector math, no LLM involved.

**Today's goal:** stop stopping at "here are some relevant chunks" and actually get an answer. We'll take retrieved chunks, stuff them into a prompt alongside the user's question, send that to Gemini, and get back a real natural-language answer grounded in your documents. This is the full RAG loop, end to end, for the first time.

---

## Step 1 — Install Dependencies (same as Day 2)

In [17]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Setup API Key

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

API key loaded successfully.


## Step 3 — Imports

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## Step 4 — Reload the Existing Vector Store

This is the payoff of persisting on Day 2 — no re-loading PDFs, no re-splitting, no re-embedding. We just point Chroma at the folder it already saved to disk.

In [26]:
PERSIST_DIR = "chroma_db"

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

vector_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings_model
)

print(f"Reloaded vector store with {vector_store._collection.count()} chunks")

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Reloaded vector store with 13 chunks


## Step 5 — Initialize the Chat Model

Same Gemini 2.5 Flash model from your email agent project. Note: **no `bind_tools()` here** — Day 1's project used tool calling (model picks a function to run). RAG doesn't need that; the model's only job here is to read context and write an answer.

In [27]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

## Step 6 — Design the Prompt Template

This is the most important piece of a RAG system's *quality*, not just its plumbing. The prompt tells the model exactly how to use the retrieved context — and critically, instructs it to **say it doesn't know rather than guess** when the context doesn't cover the question. Without this instruction, LLMs will confidently make things up (hallucinate) instead of admitting a gap.

`{context}` and `{question}` are placeholders — LangChain fills them in at run time.

In [28]:
RAG_PROMPT = """You are a helpful assistant answering questions using ONLY the context provided below.

Rules:
- Base your answer strictly on the context. Do not use outside knowledge.
- If the context does not contain enough information to answer, say so clearly instead of guessing.
- Be concise and direct.
- If helpful, mention which part of the context you drew the answer from.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

## Step 7 — Helper: Format Retrieved Chunks Into a Single Context String

The retriever returns a list of `Document` objects. The prompt template needs one plain string for `{context}`, so we join the chunks together, labeling each one by its source so the model (and you) can tell where each piece came from.

In [ ]:

def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        formatted.append(f"[Chunk {i} — {source}, page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)


## Step 8 — Build the RAG Chain (LangChain Expression Language)

This is where retrieval and generation actually connect. LangChain's `|` operator chains steps together — each step's output feeds into the next step's input, left to right:

```
question
  → {context: retriever fetches + formats chunks, question: passed through unchanged}
  → prompt (fills in the template)
  → llm (generates a response)
  → StrOutputParser() (extracts plain text from the model's response object)
```

`RunnablePassthrough()` simply means "take the input as-is and pass it along unchanged" — here it's used so the original question reaches the prompt template even while `context` is being built from the retriever in parallel.

In [30]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built.")

RAG chain built.


## Step 9 — Ask Real Questions

Now the moment this whole 3-day build has been leading to: ask a real question and get a grounded answer generated from your own documents.

In [31]:
def ask(question: str):
    print(f'Q: {question}\n')
    answer = rag_chain.invoke(question)
    print(f'A: {answer}')
    return answer

ask("What is the main topic of this document?")

Q: What is the main topic of this document?

A: The document discusses several main topics:
1.  **Cloud Computing and Modern Software Development**, including its transformation of software development, reduction of infrastructure costs, and scalability, as well as the three primary cloud service models (IaaS, PaaS, SaaS) and cloud-native technologies (Chunk 1, Chunk 3).
2.  **Artificial Intelligence and Its Impact on Modern Society**, covering its influence on businesses and communication, the definition of AI, and the reasons for its widespread adoption, such as increased computational power and large datasets (Chunk 2).
3.  **The Importance of Data Structures and Algorithms** as the foundation of efficient software development (Chunk 3).


'The document discusses several main topics:\n1.  **Cloud Computing and Modern Software Development**, including its transformation of software development, reduction of infrastructure costs, and scalability, as well as the three primary cloud service models (IaaS, PaaS, SaaS) and cloud-native technologies (Chunk 1, Chunk 3).\n2.  **Artificial Intelligence and Its Impact on Modern Society**, covering its influence on businesses and communication, the definition of AI, and the reasons for its widespread adoption, such as increased computational power and large datasets (Chunk 2).\n3.  **The Importance of Data Structures and Algorithms** as the foundation of efficient software development (Chunk 3).'

In [32]:
# Try a couple more — swap these for questions actually relevant to your data/ files
ask("Can you summarize the key points in a few sentences?")

Q: Can you summarize the key points in a few sentences?

A: The provided context discusses three main areas: Artificial Intelligence (AI), Cloud Computing, and Data Structures and Algorithms (DSA). AI is defined as machines performing human-like tasks, driven by increased computational power and large datasets, and impacting businesses and communication. Cloud computing has transformed software development by offering rented computing resources, reducing costs, and providing scalability through models like IaaS, PaaS, and SaaS, with serverless computing being useful for unpredictable workloads. Lastly, Data Structures and Algorithms are presented as the foundation of efficient software development, crucial for tasks such as searching, sorting, and memory management. (Based on Chunk 1, Chunk 2, and Chunk 3)


'The provided context discusses three main areas: Artificial Intelligence (AI), Cloud Computing, and Data Structures and Algorithms (DSA). AI is defined as machines performing human-like tasks, driven by increased computational power and large datasets, and impacting businesses and communication. Cloud computing has transformed software development by offering rented computing resources, reducing costs, and providing scalability through models like IaaS, PaaS, and SaaS, with serverless computing being useful for unpredictable workloads. Lastly, Data Structures and Algorithms are presented as the foundation of efficient software development, crucial for tasks such as searching, sorting, and memory management. (Based on Chunk 1, Chunk 2, and Chunk 3)'

In [33]:
# Test the hallucination guardrail — ask something your documents almost certainly do NOT cover
ask("What is the capital of France?")

Q: What is the capital of France?

A: The context provided does not contain information about the capital of France.


'The context provided does not contain information about the capital of France.'

## Step 10 — Inspect What Actually Got Sent to the Model

It's worth seeing the *exact* context block the model received, so RAG stops feeling like a black box.

In [34]:
sample_question = "What is the main topic of this document?"
retrieved_docs = retriever.invoke(sample_question)
context_str = format_docs(retrieved_docs)

print("--- Exact context sent to the model ---\n")
print(context_str[:1500])

--- Exact context sent to the model ---

[Chunk 1 — data\Untitled document.pdf, page 1]
------------------------------------------------------------   Cloud  Computing  and  Modern  Software  Development   Cloud  computing  has  transformed  how  software  applications  are  developed,  deployed,  and  
maintained.
 
Instead
 
of
 
purchasing
 
expensive
 
physical
 
servers,
 
organizations
 
can
 
rent
 
computing
 
resources
 
from
 
cloud
 
providers
 
on
 
demand.
 
This
 
approach
 
reduces
 
infrastructure
 
costs
 
and
 
provides
 
scalability
 
that
 
traditional
 
systems
 
cannot
 
easily
 
achieve.
  There  are  three  primary  cloud  service  models:  Infrastructure  as  a  Service  (IaaS),  Platform  as  a  
Service
 
(PaaS),
 
and
 
Software
 
as
 
a
 
Service
 
(SaaS).
 
IaaS
 
provides
 
virtual
 
machines,
 
storage,
 
and
 
networking
 
resources.
 
PaaS
 
offers
 
development
 
platforms
 
where
 
developers
 
can
 
build
 
and
 
deploy
 
applications
 
without
 
ma

---
## Day 3 Wrap-Up

Today you completed the **full RAG loop** for the first time:

`Question → Retrieve top-k chunks → Format into context → Fill prompt template → Gemini generates → Parsed text answer`

Compare this to Day 1's email agent: there, the model *decided which function to call*. Here, the model never decides anything about retrieval — that part is deterministic code (`retriever | format_docs`). The model's only job is the final generation step, using context we controlled and fed it. This is an important distinction to be able to explain: **RAG is a pipeline, tool-calling agents are a decision loop** — and later projects often combine both.

**What's still missing:**
- No visibility into *how good* the retrieval was beyond eyeballing it — no similarity score filtering yet.
- No memory — every `ask()` call is independent; the model has no idea what you asked two questions ago.
- Answers don't cite sources in a structured way yet (we saw the raw chunks in Step 10, but the model isn't instructed to output them cleanly).

**Tomorrow (Day 4):** we improve retrieval quality — tune chunk size/overlap based on what you're seeing, add similarity score thresholds so weak matches get filtered out, and get the model to cite sources properly in its answers.

See `README_Day3.md` for the full write-up of today's concepts.